# Strategy B — hierarchical classification

This notebook implements the classification-only inverse-design strategy. The first stage identifies the material family (NA, MW, or OW); the second stage classifies the additive percentage for MW or OW. The input descriptors are the flexural modulus $E_f$, flexural strength $\sigma_r$, and strain at break $\varepsilon_r$.


In [ ]:
%pip install -q imbalanced-learn openpyxl


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
# Consistent plotting style for publication-ready figures.
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.titlesize": 14,
    "axes.linewidth": 0.8,
})

FIG_DIR = Path("figures_pdf")
FIG_DIR.mkdir(exist_ok=True)


def save_pdf(fig, filename):
    fig.savefig(
        FIG_DIR / f"{filename}.pdf",
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.02,
    )


from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    learning_curve,
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from imblearn.pipeline import Pipeline as ImbPipeline


In [ ]:
# Locate the dataset in common repository, local, or Colab layouts.
candidate_paths = [
    Path("../dataset/dataset_additives.xlsx"),
    Path("dataset/dataset_additives.xlsx"),
    Path("dataset_additives.xlsx"),
    Path("/content/dataset_additives.xlsx"),
]

dataset_path = next((path for path in candidate_paths if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Dataset not found. Place dataset_additives.xlsx next to the notebook "
        "or in a dataset/ directory."
    )

print(f"Using dataset: {dataset_path}")


In [ ]:
# Load the five-column specimen-level dataset and assign concise internal names.
df_raw = pd.read_excel(dataset_path, keep_default_na=False)

column_map = {
    "material": "material",
    "percentage (%)": "percentage",
    "flexural_modulus (GPa)": "flexural_modulus",
    "flexural_strength_at_break (MPa)": "flexural_strength",
    "strain_at_break (%)": "strain_at_break",
}

missing_columns = [name for name in column_map if name not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df_raw[list(column_map)].rename(columns=column_map).copy()
numeric_columns = [
    "percentage",
    "flexural_modulus",
    "flexural_strength",
    "strain_at_break",
]
for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )

df["material"] = df["material"].astype(str).str.strip().str.upper()

if df[numeric_columns].isna().any().any():
    raise ValueError("The required numeric columns contain missing or invalid values.")

display(df.head())
print("Dataset shape:", df.shape)
print("Samples by material:")
print(df["material"].value_counts(dropna=False))


In [ ]:
X_material = df[["flexural_modulus", "flexural_strength", "strain_at_break"]].copy()
y_material = df["material"].copy()

material_encoder = LabelEncoder()
y_material_enc = material_encoder.fit_transform(y_material)

print("Material mapping:")
for cls, code in zip(material_encoder.classes_, material_encoder.transform(material_encoder.classes_)):
    print(f"{cls} -> {code}")

In [ ]:
X_train_mat, X_test_mat, y_train_mat, y_test_mat = train_test_split(
    X_material,
    y_material_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_material_enc
)

print("Train shape:", X_train_mat.shape)
print("Test shape: ", X_test_mat.shape)

In [ ]:
material_models = {
    "RandomForestClassifier": ImbPipeline([
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "LogisticRegression": ImbPipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "SVC": ImbPipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ])
}

In [ ]:
cv_material = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

material_results = []
best_material_name = None
best_material_score = -np.inf

print("=== MATERIAL CLASSIFIERS - CV ON TRAIN SET ===")

for name, base_model in material_models.items():
    model = clone(base_model)

    cv_res = cross_validate(
        model,
        X_train_mat,
        y_train_mat,
        cv=cv_material,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "f1_macro": "f1_macro"
        },
        return_train_score=True,
        n_jobs=-1,
        error_score=np.nan
    )

    row = {
        "classifier": name,
        "cv_train_accuracy_mean": np.nanmean(cv_res["train_accuracy"]),
        "cv_val_accuracy_mean": np.nanmean(cv_res["test_accuracy"]),
        "cv_val_accuracy_std": np.nanstd(cv_res["test_accuracy"]),
        "cv_val_bal_accuracy_mean": np.nanmean(cv_res["test_balanced_accuracy"]),
        "cv_val_f1_macro_mean": np.nanmean(cv_res["test_f1_macro"]),
        "cv_gap_mean": np.nanmean(cv_res["train_accuracy"] - cv_res["test_accuracy"])
    }

    material_results.append(row)

    print(
        f"{name:25} -> "
        f"CV acc={row['cv_val_accuracy_mean']:.4f} ± {row['cv_val_accuracy_std']:.4f} | "
        f"BalAcc={row['cv_val_bal_accuracy_mean']:.4f} | "
        f"F1macro={row['cv_val_f1_macro_mean']:.4f} | "
        f"Gap={row['cv_gap_mean']:.4f}"
    )

    score_for_selection = row["cv_val_f1_macro_mean"]

    if not np.isnan(score_for_selection) and score_for_selection > best_material_score:
        best_material_score = score_for_selection
        best_material_name = name

material_results_df = pd.DataFrame(material_results).sort_values(
    ["cv_val_f1_macro_mean", "cv_val_bal_accuracy_mean", "cv_val_accuracy_mean"],
    ascending=False
)

print("\nBEST MATERIAL CLASSIFIER:", best_material_name)
display(material_results_df)

In [ ]:
best_material_model = clone(material_models[best_material_name])
best_material_model.fit(X_train_mat, y_train_mat)

best_material_pred = best_material_model.predict(X_test_mat)

material_labels = list(range(len(material_encoder.classes_)))
material_class_names = [str(x) for x in material_encoder.classes_]

test_material_accuracy = accuracy_score(y_test_mat, best_material_pred)
test_material_bal_accuracy = balanced_accuracy_score(y_test_mat, best_material_pred)
test_material_f1_macro = f1_score(y_test_mat, best_material_pred, average="macro")

print("=== MATERIAL TEST RESULTS ===")
print(f"Accuracy:          {test_material_accuracy:.4f}")
print(f"Balanced accuracy: {test_material_bal_accuracy:.4f}")
print(f"Macro F1:          {test_material_f1_macro:.4f}")

print("\n=== MATERIAL CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_test_mat,
        best_material_pred,
        labels=material_labels,
        target_names=material_class_names,
        zero_division=0
    )
)

cm_mat = confusion_matrix(
    y_test_mat,
    best_material_pred,
    labels=material_labels
)

disp_mat = ConfusionMatrixDisplay(
    confusion_matrix=cm_mat,
    display_labels=material_class_names
)

fig, ax = plt.subplots(figsize=(6, 6))
disp_mat.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)

ax.set_title(f"")
ax.tick_params(axis="x", rotation=30)

fig.tight_layout()
save_pdf(fig, "fig01_confusion_matrix_material")
plt.show()


In [ ]:
cv_curve_material = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    clone(material_models[best_material_name]),
    X_material,
    y_material_enc,
    cv=cv_curve_material,
    scoring="accuracy",
    n_jobs=-1,
    train_sizes=np.linspace(0.4, 1.0, 5),
    error_score=np.nan
)

train_mean = np.nanmean(train_scores, axis=1)
train_std = np.nanstd(train_scores, axis=1)
val_mean = np.nanmean(val_scores, axis=1)
val_std = np.nanstd(val_scores, axis=1)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(train_sizes, train_mean, marker="o", label="Training accuracy")
ax.plot(train_sizes, val_mean, marker="s", label="Validation accuracy")

ax.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.2
)

ax.fill_between(
    train_sizes,
    val_mean - val_std,
    val_mean + val_std,
    alpha=0.2
)

ax.set_xlabel("Training set size")
ax.set_ylabel("Accuracy")
ax.set_title(f"")
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle=":")
ax.legend()

fig.tight_layout()
save_pdf(fig, "fig02_learning_curve_material")
plt.show()


In [ ]:
def train_percentage_classifiers(df_sub, material_name):
    X = df_sub[["flexural_modulus", "flexural_strength", "strain_at_break"]].copy()
    y = df_sub["percentage"].copy()

    pct_encoder = LabelEncoder()
    y_enc = pct_encoder.fit_transform(y)

    class_counts = pd.Series(y_enc).value_counts().sort_index()
    min_class_count = class_counts.min()

    print(f"\n=== {material_name} - CLASS COUNTS ===")
    for cls_idx, count in class_counts.items():
        original_label = pct_encoder.inverse_transform([cls_idx])[0]
        print(f"Class {original_label}: {count}")

    if min_class_count < 2:
        raise ValueError(
            f"Material {material_name}: at least one percentage class has fewer than 2 samples."
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_enc,
        test_size=0.2,
        random_state=42,
        stratify=y_enc
    )

    # Limit the number of folds to the smallest class in the training set.
    train_class_counts = pd.Series(y_train).value_counts()
    min_train_class_count = int(train_class_counts.min())

    if min_train_class_count < 2:
        raise ValueError(
            f"Material {material_name}: after train/test split, at least one class has fewer than 2 train samples."
        )

    n_splits_safe = min(5, min_train_class_count)

    classifiers = {
        "RandomForestClassifier": ImbPipeline([
            ("model", RandomForestClassifier(
                n_estimators=200,
                max_depth=5,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42
            ))
        ]),

        "LogisticRegression": ImbPipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            ))
        ]),

        "SVC": ImbPipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(
                probability=True,
                class_weight="balanced",
                random_state=42
            ))
        ])
    }

    cv = StratifiedKFold(
        n_splits=n_splits_safe,
        shuffle=True,
        random_state=42
    )

    rows = []
    best_name = None
    best_score = -np.inf

    for clf_name, base_model in classifiers.items():
        clf = clone(base_model)

        cv_res = cross_validate(
            clf,
            X_train,
            y_train,
            cv=cv,
            scoring={
                "accuracy": "accuracy",
                "balanced_accuracy": "balanced_accuracy",
                "f1_macro": "f1_macro"
            },
            return_train_score=True,
            n_jobs=-1,
            error_score=np.nan
        )

        row = {
            "classifier": clf_name,
            "cv_train_accuracy_mean": np.nanmean(cv_res["train_accuracy"]),
            "cv_val_accuracy_mean": np.nanmean(cv_res["test_accuracy"]),
            "cv_val_accuracy_std": np.nanstd(cv_res["test_accuracy"]),
            "cv_val_bal_accuracy_mean": np.nanmean(cv_res["test_balanced_accuracy"]),
            "cv_val_f1_macro_mean": np.nanmean(cv_res["test_f1_macro"]),
            "cv_gap_mean": np.nanmean(cv_res["train_accuracy"] - cv_res["test_accuracy"])
        }

        rows.append(row)

        score_for_selection = row["cv_val_f1_macro_mean"]
        if not np.isnan(score_for_selection) and score_for_selection > best_score:
            best_score = score_for_selection
            best_name = clf_name

    results_df = pd.DataFrame(rows).sort_values(
        ["cv_val_f1_macro_mean", "cv_val_bal_accuracy_mean", "cv_val_accuracy_mean"],
        ascending=False
    )

    trained_models = {}
    for clf_name, base_model in classifiers.items():
        clf = clone(base_model)
        clf.fit(X_train, y_train)
        trained_models[clf_name] = clf

    best_model = trained_models[best_name]
    y_test_pred = best_model.predict(X_test)

    percentage_labels = list(range(len(pct_encoder.classes_)))
    percentage_class_names = [str(x) for x in pct_encoder.classes_]

    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_bal_accuracy = balanced_accuracy_score(y_test, y_test_pred)
    test_f1_macro = f1_score(y_test, y_test_pred, average="macro")

    print(f"\n=== PERCENTAGE CLASSIFICATION - {material_name} ===")
    print("Best classifier:", best_name)
    print(f"CV folds used:           {n_splits_safe}")
    print(f"Test accuracy:           {test_accuracy:.4f}")
    print(f"Test balanced accuracy:  {test_bal_accuracy:.4f}")
    print(f"Test macro F1:           {test_f1_macro:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_test_pred,
            labels=percentage_labels,
            target_names=percentage_class_names,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_test,
        y_test_pred,
        labels=percentage_labels
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=percentage_class_names
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)

    ax.set_title(f"")
    ax.tick_params(axis="x", rotation=30)

    fig.tight_layout()
    save_pdf(fig, f"fig03_confusion_matrix_percentage_{material_name.lower()}")
    plt.show()

    return {
        "material_name": material_name,
        "X": X,
        "y": y,
        "encoder": pct_encoder,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "classifiers": classifiers,
        "trained_models": trained_models,
        "best_name": best_name,
        "best_model": best_model,
        "best_test_pred": y_test_pred,
        "results_df": results_df,
        "test_accuracy": test_accuracy,
        "test_bal_accuracy": test_bal_accuracy,
        "test_f1_macro": test_f1_macro,
        "n_splits_used": n_splits_safe
    }


In [ ]:
df_MW = df[df["material"] == "MW"].copy()
df_OW = df[df["material"] == "OW"].copy()
df_NA = df[df["material"] == "NA"].copy()

print("MW shape:", df_MW.shape)
print("OW shape:", df_OW.shape)
print("NA shape:", df_NA.shape)

print("\nMW percentage classes:", sorted(df_MW["percentage"].unique()))
print("OW percentage classes:", sorted(df_OW["percentage"].unique()))
print("NA percentage classes:", sorted(df_NA["percentage"].unique()))

In [ ]:
mw_percentage_system = train_percentage_classifiers(df_MW, "MW")
ow_percentage_system = train_percentage_classifiers(df_OW, "OW")

In [ ]:
print("=== MW PERCENTAGE CLASSIFICATION RESULTS ===")
display(mw_percentage_system["results_df"])

print("\n=== OW PERCENTAGE CLASSIFICATION RESULTS ===")
display(ow_percentage_system["results_df"])

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np
import pandas as pd

def check_classification_overfitting(system_name, pct_system):

    X_train = pct_system["X_train"]
    y_train = pct_system["y_train"]

    best_name = pct_system["best_name"]
    best_model = clone(pct_system["trained_models"][best_name])

    min_train_class_count = int(pd.Series(y_train).value_counts().min())
    n_splits_safe = min(5, min_train_class_count)

    if n_splits_safe < 2:
        raise ValueError(
            f"{system_name}: at least one class has fewer than 2 train samples."
        )

    cv = StratifiedKFold(
        n_splits=n_splits_safe,
        shuffle=True,
        random_state=42
    )

    scores = cross_validate(
        best_model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=-1
    )

    # Mean training and validation scores.
    train_acc_mean = np.mean(scores["train_accuracy"])
    val_acc_mean = np.mean(scores["test_accuracy"])

    train_f1_mean = np.mean(scores["train_f1_macro"])
    val_f1_mean = np.mean(scores["test_f1_macro"])

    # Absolute and relative generalization gaps.
    acc_gap_abs = train_acc_mean - val_acc_mean
    acc_gap_rel = acc_gap_abs / train_acc_mean if train_acc_mean > 0 else np.nan

    f1_gap_abs = np.mean(scores["train_f1_macro"] - scores["test_f1_macro"])
    f1_gap_rel = f1_gap_abs / val_f1_mean if val_f1_mean > 0 else np.nan

    # Relative variability of the validation F1 score.
    val_f1_std = np.std(scores["test_f1_macro"])
    stability = val_f1_std / val_f1_mean if val_f1_mean > 0 else np.nan

    # Qualitative overfitting indicator based on the relative F1 gap.
    if f1_gap_rel < 0.10:
        risk = "low"
    elif f1_gap_rel < 0.25:
        risk = "moderate"
    else:
        risk = "high"

    return {
        "system": system_name,
        "best_classifier": best_name,
        "cv_folds_used": n_splits_safe,

        "train_acc_mean": train_acc_mean,
        "val_acc_mean": val_acc_mean,

        "train_f1_mean": train_f1_mean,
        "val_f1_mean": val_f1_mean,

        "acc_gap_abs": acc_gap_abs,
        "acc_gap_rel": acc_gap_rel,

        "f1_gap_abs": f1_gap_abs,
        "f1_gap_rel": f1_gap_rel,

        "val_f1_std": val_f1_std,
        "stability": stability,

        "overfitting_risk": risk
    }


In [ ]:
def print_overfitting_report(report):
    print(f"System: {report['system']}")
    print(f"Best model: {report['best_classifier']}")
    print(f"Train ACC: {report['train_acc_mean']:.3f}")
    print(f"Val ACC:   {report['val_acc_mean']:.3f}")
    print(f"Train F1:  {report['train_f1_mean']:.3f}")
    print(f"Val F1:    {report['val_f1_mean']:.3f}")
    print(f"ACC gap:   {report['acc_gap_abs']:.3f} ({report['acc_gap_rel']*100:.1f}%)")
    print(f"F1 gap:    {report['f1_gap_abs']:.3f} ({report['f1_gap_rel']*100:.1f}%)")
    print(f"Stability: {report['stability']:.3f}")
    print(f"Risk:      {report['overfitting_risk']}")
    print("-" * 40)


# Check overfitting for percentage classification models
mw_overfit_df = check_classification_overfitting("MW", mw_percentage_system)
ow_overfit_df = check_classification_overfitting("OW", ow_percentage_system)


print("=== OVERFITTING CHECK - MW PERCENTAGE ===")
print_overfitting_report(mw_overfit_df)

print("=== OVERFITTING CHECK - OW PERCENTAGE ===")
print_overfitting_report(ow_overfit_df)

In [ ]:
def run_percentage_cv(df_sub, pct_system, title):
    X = df_sub[["flexural_modulus", "flexural_strength", "strain_at_break"]].copy()
    y = df_sub["percentage"].copy()

    encoder = LabelEncoder()
    y_enc = encoder.fit_transform(y)

    n_splits_safe = pct_system["n_splits_used"]

    cv = StratifiedKFold(
        n_splits=n_splits_safe,
        shuffle=True,
        random_state=42
    )

    best_name = pct_system["best_name"]
    best_pipeline = clone(pct_system["classifiers"][best_name])

    cv_results = cross_validate(
        best_pipeline,
        X,
        y_enc,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "f1_macro": "f1_macro"
        },
        return_train_score=True,
        n_jobs=-1,
        error_score=np.nan
    )

    cv_summary = pd.DataFrame({
        "fold": range(1, n_splits_safe + 1),
        "train_accuracy": cv_results["train_accuracy"],
        "val_accuracy": cv_results["test_accuracy"],
        "val_bal_accuracy": cv_results["test_balanced_accuracy"],
        "val_f1_macro": cv_results["test_f1_macro"]
    })

    print(f"=== CROSS-VALIDATION - {title} ===")
    display(cv_summary)
    print("Mean train accuracy:", np.nanmean(cv_summary["train_accuracy"]))
    print("Mean val accuracy:  ", np.nanmean(cv_summary["val_accuracy"]))
    print("Mean val bal acc:   ", np.nanmean(cv_summary["val_bal_accuracy"]))
    print("Mean val macro F1:  ", np.nanmean(cv_summary["val_f1_macro"]))
    print("Mean gap:           ", np.nanmean(cv_summary["train_accuracy"] - cv_summary["val_accuracy"]))

In [ ]:
run_percentage_cv(df_MW, mw_percentage_system, "MW")
run_percentage_cv(df_OW, ow_percentage_system, "OW")

In [ ]:
def plot_percentage_learning_curve(df_sub, pct_system, title, filename):
    X = df_sub[["flexural_modulus", "flexural_strength", "strain_at_break"]].copy()
    y = df_sub["percentage"].copy()

    encoder = LabelEncoder()
    y_enc = encoder.fit_transform(y)

    best_name = pct_system["best_name"]
    best_pipeline = clone(pct_system["classifiers"][best_name])

    n_splits_safe = pct_system["n_splits_used"]

    cv_curve = StratifiedKFold(
        n_splits=n_splits_safe,
        shuffle=True,
        random_state=42
    )

    train_sizes, train_scores, val_scores = learning_curve(
        best_pipeline,
        X,
        y_enc,
        cv=cv_curve,
        scoring="accuracy",
        n_jobs=-1,
        train_sizes=np.linspace(0.4, 1.0, 5),
        error_score=np.nan
    )

    train_mean = np.nanmean(train_scores, axis=1)
    train_std = np.nanstd(train_scores, axis=1)
    val_mean = np.nanmean(val_scores, axis=1)
    val_std = np.nanstd(val_scores, axis=1)

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(train_sizes, train_mean, marker="o", label="Training accuracy")
    ax.plot(train_sizes, val_mean, marker="s", label="Validation accuracy")

    ax.fill_between(
        train_sizes,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.2
    )

    ax.fill_between(
        train_sizes,
        val_mean - val_std,
        val_mean + val_std,
        alpha=0.2
    )

    ax.set_xlabel("Training set size")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"")
    ax.set_ylim(0, 1.05)
    ax.grid(True, linestyle=":")
    ax.legend()

    fig.tight_layout()
    save_pdf(fig, filename)
    plt.show()


In [ ]:
plot_percentage_learning_curve(
    df_MW,
    mw_percentage_system,
    "MW",
    "fig04_learning_curve_percentage_mw"
)

plot_percentage_learning_curve(
    df_OW,
    ow_percentage_system,
    "OW",
    "fig05_learning_curve_percentage_ow"
)


In [ ]:
def predict_material_and_percentage_classification(
    flexural_modulus, flexural_strength, strain_at_break
):
    """Predict material family and additive percentage class from target properties."""
    X_input = pd.DataFrame(
        [[flexural_modulus, flexural_strength, strain_at_break]],
        columns=["flexural_modulus", "flexural_strength", "strain_at_break"],
    )

    print("\n=== INPUT RECEIVED ===")
    display(X_input)

    pred_material_enc = best_material_model.predict(X_input)[0]
    pred_material = material_encoder.inverse_transform([pred_material_enc])[0]

    print("\n=== MATERIAL CLASSIFICATION RESULT ===")
    print(f"Best material classifier: {best_material_name}")
    print(f"Predicted material: {pred_material}")

    if hasattr(best_material_model, "predict_proba"):
        probabilities = best_material_model.predict_proba(X_input)[0]
        print("Material class probabilities:")
        for class_index, probability in enumerate(probabilities):
            material = material_encoder.inverse_transform([class_index])[0]
            print(f"  {material}: {probability:.3f}")

    if pred_material == "NA":
        print("\nNo additive detected; predicted percentage = 0 wt.%")
        return pd.DataFrame([{
            "predicted_material": "NA",
            "material_classifier": best_material_name,
            "predicted_percentage": 0.0,
            "percentage_classifier": "Constant",
        }])

    if pred_material == "MW":
        pct_system = mw_percentage_system
    elif pred_material == "OW":
        pct_system = ow_percentage_system
    else:
        raise ValueError(f"Unsupported predicted material: {pred_material}")

    best_pct_name = pct_system["best_name"]
    best_pct_model = pct_system["best_model"]
    pct_encoder = pct_system["encoder"]

    pred_pct_enc = best_pct_model.predict(X_input)[0]
    pred_pct = pct_encoder.inverse_transform([pred_pct_enc])[0]

    print("\n=== PERCENTAGE CLASSIFICATION RESULT ===")
    print(f"Best percentage classifier for {pred_material}: {best_pct_name}")
    print(f"Predicted percentage: {pred_pct} wt.%")

    if hasattr(best_pct_model, "predict_proba"):
        probabilities = best_pct_model.predict_proba(X_input)[0]
        print("Percentage-class probabilities:")
        for class_index, probability in enumerate(probabilities):
            pct_class = pct_encoder.inverse_transform([class_index])[0]
            print(f"  {pct_class}: {probability:.3f}")

    return pd.DataFrame([{
        "predicted_material": pred_material,
        "material_classifier": best_material_name,
        "predicted_percentage": pred_pct,
        "percentage_classifier": best_pct_name,
    }])


In [ ]:
results_input = predict_material_and_percentage_classification(
    flexural_modulus=3.89,      # E_f [GPa]
    flexural_strength=47.46,  # sigma_r [MPa]
    strain_at_break=1.80,  # epsilon_r [%]
)

display(results_input)


In [ ]:
results_input = predict_material_and_percentage_classification(
    flexural_modulus=3.70,      # E_f [GPa]
    flexural_strength=100.00,  # sigma_r [MPa]
    strain_at_break=4.80,  # epsilon_r [%]
)

display(results_input)


In [ ]:
results_input = predict_material_and_percentage_classification(
    flexural_modulus=2.10,      # E_f [GPa]
    flexural_strength=40.46,  # sigma_r [MPa]
    strain_at_break=3.00,  # epsilon_r [%]
)

display(results_input)


In [ ]:
# Optional: create a zip archive of the generated PDF figures.
# This cell is safe for local/Jupyter execution and does not require Google Colab.

import shutil
from pathlib import Path

figures_dir = Path("figures_pdf")
if figures_dir.exists():
    shutil.make_archive("figures_pdf", "zip", figures_dir)
    print("Created figures_pdf.zip")
else:
    print("No figures_pdf folder found.")
